# Single-Task Preclinical Toxicity Prediction with GIN and Avalon Fingerprints: Tox21 NR-AR

This notebook trains a GIN–Avalon hybrid model to predict activity for the Tox21 NR-AR assay. It combines two molecular representations: a graph representation processed by a Graph Isomorphism Network (GIN), and a 1,024-bit Avalon fingerprint processed alongside the graph embedding by a multilayer perceptron (MLP).

The workflow covers data preparation, molecular representation, graph construction, hyperparameter optimization, and held-out test evaluation. Samples with missing NR-AhR labels are removed before training. 

## 1. Imports 

The notebook uses RDKit for molecular processing and Avalon fingerprints, PyTorch and PyTorch Geometric for model training, scikit-learn for data splitting and metrics, and Optuna for hyperparameter search.

In [ ]:
from rdkit import Chem
import pandas as pd 
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from rdkit.Avalon.pyAvalonTools import GetAvalonFP
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool
import optuna
from optuna.trial import TrialState


In [ ]:
device = torch.device("cpu")
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

## 2. Load and clean the Tox21 dataset

The CSV is loaded into a pandas DataFrame. SMILES strings are stripped of surrounding whitespace and validated with RDKit. Invalid strings and molecules with one or zero atoms are removed.

In [ ]:
df = pd.read_csv("tox21.csv")

In [ ]:
def clean_and_validate_smiles(smi):
    if not isinstance(smi, str):
        return None
    smi = smi.strip().replace(" ", "")
    if "." in smi:
        smi = smi.split(".")[0]
    mol = Chem.MolFromSmiles(smi)
    if mol is not None and mol.GetNumAtoms() > 1:
        return smi
    return None

df["smiles"] = df["smiles"].apply(clean_and_validate_smiles)
df = df[df["smiles"].notna()].reset_index(drop=True)
print("Valid SMILES:", len(df))


# 3. Define the target toxicity endpoint

This is a single-task experiment: NR-AR is the only prediction target in this notebook. The assay labels are extracted as numeric values, where 1 indicates an active response and 0 indicates an inactive response. Missing assay labels are identified separately and removed before graph creation and model training.

In [ ]:
target_col = "NR-AR"
labels = df[target_col].values.astype(float)


## 4. Inspect the target-label distribution 

Count positive, negative, and missing NR-AR labels. This provides a quick view of class imbalance and label availability before filtering missing targets.

In [ ]:
positives = np.sum(labels == 1)
negatives = np.sum(labels == 0)
nulls = np.sum(np.isnan(labels))
print("\nLabel distribution for", target_col)
print("Positives :", int(positives))
print("Negatives :", int(negatives))
print("Null values:", int(nulls))

## 5. Remove samples with missing NR-AR labels

Rows without an observed NR-AR label are removed because the single-task binary loss requires a known target. The corresponding SMILES and labels are kept aligned for feature generation.

In [ ]:
# Remove NaNs
valid_mask = ~np.isnan(labels)
df_clean = df[valid_mask].reset_index(drop=True)
labels = labels[valid_mask]

smiles_list = df_clean["smiles"].tolist()


In [ ]:
print("Original rows:", len(df))
print("Cleaned rows :", len(df_clean))
print("NaNs removed :", len(df) - len(df_clean))

## 6. Generate Avalon molecular fingerprints

Convert each cleaned molecule into a 1,024-bit Avalon fingerprint. The fingerprint is a fixed-length vector encoding molecular substructure information and serves as a complementary representation to the molecular graph.

In [ ]:
def get_avalon_fp(smi, n_bits=1024):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)
    fp = GetAvalonFP(mol, nBits=n_bits)
    return np.array(fp, dtype=np.float32)

smiles_list = df_clean["smiles"].tolist()
avalon_fps = np.array([get_avalon_fp(s) for s in smiles_list], dtype=np.float32)


In [ ]:
ATOM_LIST = list(range(1, 119))

def atom_features(atom):
    Z = atom.GetAtomicNum()
    onehot = [int(Z == i) for i in ATOM_LIST]
    return np.array(onehot, dtype=np.float32)


## 7. Convert molecules into graph data

Convert each molecule into a PyTorch Geometric Data object. Atoms become nodes with one-hot features; each chemical bond becomes two directed edges so messages can pass in both directions. Each graph also stores its NR-AR label and corresponding Avalon fingerprint.

In [ ]:
def mol_to_pyg(smiles, avalon_fp, y_val):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None

    # Node features
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()],
                     dtype=torch.float32)

    # Edges
    edge_index = []
    for bond in mol.GetBonds():
        a = bond.GetBeginAtomIdx()
        b = bond.GetEndAtomIdx()
        edge_index.append([a, b])
        edge_index.append([b, a])

    if len(edge_index) == 0:
        edge_index = torch.empty((2,0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # CORRECT shapes
    y = torch.tensor(float(y_val), dtype=torch.float32)   # shape ()
    fp = torch.tensor(avalon_fp, dtype=torch.float32).unsqueeze(0)  # (1,1024)

    return Data(x=x, edge_index=edge_index, y=y, fp=fp)


In [ ]:
pyg_data_list = []
for smi, fp, y in zip(smiles_list, avalon_fps, labels):
    d = mol_to_pyg(smi, fp, y)
    if d is not None:
        pyg_data_list.append(d)

print("Total graphs:", len(pyg_data_list))
node_dim = pyg_data_list[0].x.shape[1]


## 8. Split data and create mini-batches

Split graph indices into training, validation, and test subsets using a fixed random state. The first split reserves 20% for testing; 20% of the remaining 80% is then assigned to validation (approximately 64% train, 16% validation, 20% test). Mini-batches are created with PyTorch Geometric loaders.

In [ ]:
indices = list(range(len(pyg_data_list)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=42)

def make_loader(idx_list, batch_size, shuffle):
    data = [pyg_data_list[i] for i in idx_list]
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle)

train_loader_base = make_loader(train_idx, 32, True)
val_loader_base   = make_loader(val_idx,   32, False)


## 9. Define the GIN graph encoder

The graph encoder applies two GINConv layers with multilayer perceptrons and ReLU activations. global_mean_pool aggregates node embeddings into a fixed-size graph representation for each molecule.

In [ ]:
class GINEncoder(nn.Module):
    def __init__(self, node_dim, gin_hidden_dim=128, graph_dim=128):
        super().__init__()
        nn1 = nn.Sequential(
            nn.Linear(node_dim, gin_hidden_dim),
            nn.ReLU(),
            nn.Linear(gin_hidden_dim, gin_hidden_dim)
        )
        self.conv1 = GINConv(nn1)

        nn2 = nn.Sequential(
            nn.Linear(gin_hidden_dim, gin_hidden_dim),
            nn.ReLU(),
            nn.Linear(gin_hidden_dim, graph_dim)
        )
        self.conv2 = GINConv(nn2)

    def forward(self, x, edge_index, batch):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        return global_mean_pool(h, batch)


## 10. Combine GIN and Avalon representations

The hybrid model concatenates the graph embedding from the GIN encoder with the 1,024-bit Avalon fingerprint. A feed-forward MLP with dropout layers processes the combined representation and outputs one logit per molecule for NR-AR classification.

In [ ]:
class CombinedGINAvalon(nn.Module):
    def __init__(self, node_dim, avalon_dim=1024,
                 gin_hidden_dim=128, graph_dim=128,
                 hidden1=512, hidden2=256, dropout=0.3):
        super().__init__()
        self.gnn = GINEncoder(node_dim, gin_hidden_dim, graph_dim)
        self.mlp = nn.Sequential(
            nn.Linear(avalon_dim + graph_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1)
        )

    def forward(self, data):
        g_emb = self.gnn(data.x, data.edge_index, data.batch)  # (B,gdim)
        fp = data.fp                                           # (B,1024)
        z = torch.cat([fp, g_emb], dim=-1)
        return self.mlp(z).view(-1)  # (B,)


## 11. Set loss function and train model for one epoch

BCEWithLogitsLoss() calculates the binary classification loss. It combines a sigmoid operation with binary cross-entropy in a numerically stable way.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

criterion = nn.BCEWithLogitsLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def eval_model(model, loader):
    model.eval()
    all_probs = []
    all_y = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = torch.sigmoid(model(batch))
            all_probs.append(pred.cpu().numpy())
            all_y.append(batch.y.cpu().numpy())
    return np.concatenate(all_y), np.concatenate(all_probs)


## 12. Hyperparameter optimization with Optuna

Optuna is used to identify a suitable combination of hyperparameters for the GIN–Avalon hybrid model. Each trial samples a set of hyperparameters from predefined search spaces, initializes a new model, and trains it on the training dataset.

In [ ]:
def objective(trial):
    gin_hidden_dim = trial.suggest_categorical("gin_hidden_dim", [64,128,192])
    graph_dim      = trial.suggest_categorical("graph_dim",     [64,128,192])
    hidden1        = trial.suggest_categorical("hidden1",       [256,512,768,1024])
    hidden2        = trial.suggest_categorical("hidden2",       [128,256,384,512])
    dropout        = trial.suggest_float("dropout", 0.1, 0.5)
    lr             = trial.suggest_float("lr", 1e-4, 2e-3, log=True)
    batch_size     = trial.suggest_categorical("batch_size", [16,32,64])
    epochs         = trial.suggest_int("epochs", 20, 40)

    train_loader = make_loader(train_idx, batch_size, True)
    val_loader   = make_loader(val_idx, batch_size, False)

    model = CombinedGINAvalon(
        node_dim=node_dim,
        gin_hidden_dim=gin_hidden_dim,
        graph_dim=graph_dim,
        hidden1=hidden1,
        hidden2=hidden2,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        train_one_epoch(model, train_loader, optimizer)

    y_true, y_prob = eval_model(model, val_loader)
    return roc_auc_score(y_true, y_prob)


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best AUC:", study.best_value)
print("Best params:", study.best_params)


In [ ]:
best_params = study.best_params
print(best_params)


## 13. Prepare Final training and test sets 

After hyperparameter optimization, the training and validation indices are combined to form the final training dataset. This allows the final model to learn from both subsets after the hyperparameters have been selected.

The test set remains separate and is used for the final performance evaluation. Data loaders are created for the combined training set and test set using the batch size selected by Optuna. Shuffling is enabled for training and disabled for testing

In [ ]:
# Combine train and validation indices
trainval_idx = train_idx + val_idx

trainval_loader = make_loader(trainval_idx, batch_size=best_params["batch_size"], shuffle=True)
test_loader     = make_loader(test_idx,     batch_size=best_params["batch_size"], shuffle=False)


The final CombinedGINAvalon model is initialized using the best hyperparameters identified by Optuna. These include the GIN hidden dimension, graph embedding dimension, MLP hidden dimensions, and dropout rate.

The Avalon fingerprint dimension is set to 1,024, corresponding to the fingerprint representation used in this notebook. The model is moved to the selected computing device, and the Adam optimizer is initialized using the best learning rate.

In [ ]:
final_model = CombinedGINAvalon(
    node_dim=node_dim,
    avalon_dim=1024,
    gin_hidden_dim=best_params["gin_hidden_dim"],
    graph_dim=best_params["graph_dim"],
    hidden1=best_params["hidden1"],
    hidden2=best_params["hidden2"],
    dropout=best_params["dropout"]
).to(device)

final_optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=best_params["lr"]
)


## 14. Define Evaluation Metrics and Threshold Selection

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

def best_f1_threshold(y_true_np, probs_np):
    """Find threshold that maximizes F1 for a SINGLE BINARY task."""
    best_t = 0.5
    best_f1 = 0.0
    for t in np.arange(0.05, 0.95, 0.05):
        preds = (probs_np >= t).astype(int)
        f1 = f1_score(y_true_np, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t


def compute_metrics(y_true_np, probs_np, threshold=0.5):
    """Compute ACC, PREC, REC, F1, AUC for a SINGLE binary classification task."""
    preds = (probs_np >= threshold).astype(int)

    acc  = accuracy_score(y_true_np, preds)
    prec = precision_score(y_true_np, preds, zero_division=0)
    rec  = recall_score(y_true_np, preds, zero_division=0)
    f1   = f1_score(y_true_np, preds, zero_division=0)
    auc  = roc_auc_score(y_true_np, probs_np)

    return acc, prec, rec, f1, auc


## 15. Train the Final Model and evaluate performance

In [ ]:
final_epochs = best_params["epochs"]

for epoch in range(1, final_epochs + 1):
    loss = train_one_epoch(final_model, trainval_loader, final_optimizer)
    if epoch % 5 == 0:
        print(f"Epoch {epoch}/{final_epochs} - Loss: {loss:.4f}")

# -----------------------------
# Evaluate on TEST set
# -----------------------------
y_true, y_prob = eval_model(final_model, test_loader)

# Choose best F1 threshold
best_t = best_f1_threshold(y_true, y_prob)

# Compute metrics
acc, prec, rec, f1, auc = compute_metrics(y_true, y_prob, threshold=best_t)

print("\n===== FINAL TEST PERFORMANCE (NR-AR) =====")
print(f"Best Threshold (F1): {best_t:.2f}")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {auc:.4f}")
